<a href="https://colab.research.google.com/github/evinracher/3008410-intelligent-systems/blob/main/week6/exercise2/adversarial_text_attacks_and_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adversarial attacks

Adversarial attacks on natural language processing models aim to expose vulnerabilities by subtly altering input texts to mislead the model’s predictions. Various attack recipes have been developed, each with unique strategies ranging from word-level synonym replacements to character-level perturbations. The table below compares some of the most popular TextAttack recipes, highlighting their key features, strengths, and typical use cases.


| Attack Name       | Description                                                | Strengths                              | Weaknesses                         | Typical Use Case                   |
|-------------------|------------------------------------------------------------|--------------------------------------|-----------------------------------|----------------------------------|
| PWWSRen2019       | Probability Weighted Word Saliency attack. Replaces important words weighted by model sensitivity. | Effective at word-level perturbations, fast | Can struggle with complex context | Text classification adversarial testing |
| TextFoolerJin2019 | Uses word embeddings to replace words with synonyms preserving semantics. | Preserves semantic meaning, intuitive | Sometimes produces unnatural sentences | Robust synonym-based attacks     |
| DeepWordBug       | Generates character-level perturbations like typos and swaps to fool models. | Effective against typo-sensitive models | Less effective on robust models   | Testing typo robustness          |
| BAEGarg2019       | Generates adversarial examples by masking and replacing words using BERT predictions. | Context-aware replacements            | Computationally expensive          | Semantic-aware adversarial attacks |
| HotFlip           | Uses gradient information to flip characters for adversarial attacks. | Gradient-guided, effective character-level attacks | Needs white-box access to gradients | White-box adversarial testing    |
| TextBugger        | Combines character and word-level perturbations, including typos and synonym replacements. | Versatile, black-box attacks          | May reduce readability             | Black-box adversarial generation |

---

| Attack Name       | Original Text | Adversarial Example | Key Change |
|-------------------|--------------|--------------------|-----------|
| PWWSRen2019 | The movie was absolutely wonderful and I loved every minute of it. | The movie was absolutely **ordinary** and I loved every minute of it. | wonderful → ordinary |
| TextFoolerJin2019 | The movie was absolutely wonderful and I loved every minute of it. | The movie was absolutely **marvelous** and I **adored** every minute of it. | wonderful → marvelous, loved → adored |
| DeepWordBug | The movie was absolutely wonderful and I loved every minute of it. | The **mov1e** was **absolut3ly wond3rful** and I loved every minute of it. | character typos |


In [ ]:
!pip install \
    textattack==0.3.10 \
    transformers==4.44.2 \
    tokenizers==0.19.1 \
    datasets==3.6.0 \
    evaluate==0.4.3 \
    sentence-transformers==3.3.1 \
    textblob==0.18.0 \
    torch>=2.0.0 \
    --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


# Login to Hugging Face
This notebook uses public models and datasets, so login is not required.
If you hit rate limits while downloading, you can authenticate with a token.

Options:
- Set an environment variable `HF_TOKEN`.
- (Kaggle) Provide a JSON file at `/kaggle/input/autenti/AUTH nn.json` with an `API_KEY` field.


In [ ]:
import json
import os
from pathlib import Path

from huggingface_hub import login

# Optional: authenticate to increase rate limits when downloading models/datasets.
# For the public assets used here (GLUE SST-2 and DistilBERT SST-2), login is not required.

token = os.getenv("HF_TOKEN")

# Kaggle-specific fallback: allow reading the token from a mounted dataset, if present.
config_path = Path("/kaggle/input/datasets/reinaldolopeznarvaez/api-key/AUTH.json")
if token is None and config_path.exists():
    with config_path.open("r", encoding="utf-8") as f:
        config = json.load(f)
    token = config.get("API_KEY")

if token:
    login(token=token)
    print("Successful login to Hugging Face.")
else:
    print("No Hugging Face token found; continuing without login.")


Successful login to Hugging Face.


# Import libraries

In [ ]:
import logging
import warnings

import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    logging as hf_logging,
)

from textattack.attack_recipes import PWWSRen2019
from textattack.models.wrappers import HuggingFaceModelWrapper

# Optional (not used in the baseline code below): semantic-similarity-based signals for detection.
from sentence_transformers import SentenceTransformer, util

from textblob import TextBlob

# Print versions to make runs reproducible when sharing results.
import transformers
import datasets
import evaluate
import textattack
import sentence_transformers
import textblob


# Reduce noise in notebook output.
warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


# Adversarial Functions

In [ ]:
def preprocess_text(text: str) -> str:
    """Optional normalization step.

    NOTE: `TextBlob(text).correct()` is slow and can change semantics; keep it off unless you are
    explicitly studying spelling-correction as a defense.
    """

    return str(TextBlob(text).correct())


def generate_adversarial_examples(model_wrapper, dataset, num_examples: int = 20, verbose: bool = True):
    """Generate adversarial examples with TextAttack.

    Returns a list of dicts with keys:
    - `text`: perturbed text (or original text if the attack fails)
    - `label`: ground-truth label from the original example

    Guardrails:
    - TextAttack can emit failed/skipped results; we keep the original text so fine-tuning can proceed.
    """

    from textattack import AttackArgs, Attacker

    attack = PWWSRen2019.build(model_wrapper)
    attack_args = AttackArgs(
        num_examples=num_examples,
        shuffle=True,        # randomize which samples are attacked
        disable_stdout=True, # silence TextAttack internal prints (our prints still show)
    )
    attacker = Attacker(attack, dataset, attack_args)

    results = []
    for i, result in enumerate(attacker.attack_dataset()):
        # TextAttack results differ for success/fail/skip; guard against missing fields.
        try:
            original_text = result.original_text()
        except Exception:
            original_text = None

        try:
            perturbed_text = result.perturbed_text()
        except Exception:
            perturbed_text = None

        original_label = getattr(getattr(result, "original_result", None), "ground_truth_output", None)
        if original_label is None:
            raise ValueError("Missing ground-truth label; check that the TextAttack dataset includes labels.")

        predicted_label = getattr(getattr(result, "perturbed_result", None), "output", None)

        # If the attack fails, fall back to the original text to avoid downstream crashes.
        attack_text = perturbed_text or original_text
        if attack_text is None:
            continue

        if verbose:
            print(f"\nExample {i + 1}")
            print("Original text:", original_text)
            print("Perturbed text:", perturbed_text)
            print("Ground-truth label:", original_label)
            print("Predicted label after attack:", predicted_label)
            print("Attack successful:", original_label != predicted_label)

        results.append({
            "text": attack_text,
            "label": int(original_label),
        })

    return results


class CustomDataset(torch.utils.data.Dataset):
    """Minimal dataset wrapper for Hugging Face `Trainer`.

    Each item returns tokenized tensors + an integer label.
    """

    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]

        # Fixed-length padding simplifies batching but can waste compute; tune `max_length` as needed.
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


# Datasets
| Dataset Task | Description                                  | # Train Examples | # Validation Examples | # Test Examples     | Labels                                         |
|--------------|----------------------------------------------|------------------|-----------------------|---------------------|------------------------------------------------|
| SST-2        | Sentiment classification (positive/negative) | 67,349           | 872                   | 1,821               | 0 = negative, 1 = positive                     |
| MRPC         | Paraphrase detection                         | 3,668            | 408                   | 1,725               | 0 = not paraphrase, 1 = paraphrase             |
| QQP          | Quora Question Pairs (paraphrase detection) | 363,849          | 40,431                | 390,965             | 0 = not duplicate, 1 = duplicate               |
| QNLI         | Question-answer entailment                   | 104,743          | 5,463                 | 5,463               | 0 = not entailment, 1 = entailment             |
| MNLI         | Multi-genre Natural Language Inference      | 392,702          | 9,815 (matched)        | 9,796 (mismatched)  | 0 = contradiction, 1 = neutral, 2 = entailment |
| CoLA         | Acceptability of English sentences           | 8,551            | 1,043                 | 1,063               | 0 = unacceptable, 1 = acceptable               |
| RTE          | Recognizing Textual Entailment               | 2,490            | 277                   | 3,000               | 0 = not entailment, 1 = entailment             |
| WNLI         | Winograd Schema Challenge                    | 635              | 71                    | 146                 | 0 or 1 (coreference resolution)                |

# Models

| Feature                          | `distilbert-base-uncased-finetuned-sst-2-english`       | `cardiffnlp/twitter-roberta-base-sentiment-latest`          |
|----------------------------------|-----------------------------------------------------------|-------------------------------------------------------------|
| **Architecture**                | DistilBERT (lightweight BERT)                            | RoBERTa (Robustly optimized BERT approach)                  |
| **Pretraining Corpus**          | BooksCorpus + English Wikipedia (via BERT)               | 124M English Tweets                                         |
| **Fine-tuned On**              | SST-2 (Stanford Sentiment Treebank)                      | TweetEval sentiment task                                    |
| **Sentiment Labels**            | 0 = Negative, 1 = Positive                               | 0 = Negative, 1 = Neutral, 2 = Positive                     |
| **Domain Focus**                | General (Movie reviews, formal English)                 | Social media (Twitter-specific)                            |
| **Model Size**                  | ~66M parameters                                          | ~125M parameters                                            |
| **Tokenizer**                   | `distilbert-base-uncased` tokenizer                     | `twitter-roberta-base` tokenizer (handles hashtags, emojis)|
| **Performance (General Text)**  | Good general sentiment classification                   | Weaker on non-social media text                            |
| **Performance (Tweets)**        | Moderate, not optimized for tweets                      | Very strong—trained on tweets                              |
| **Use Case Fit**                | Academic, reviews, formal text                          | Twitter, social listening, short informal text             |


In [ ]:
def main():
    # Configuration knobs for the Activities section.
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"

    # How many examples to attack (also equals the number of adversarial examples we attempt to generate).
    attack_examples = 20

    # How many clean examples to include for adversarial training.
    clean_train_examples = 500

    # Validation subset size for quick iteration.
    val_examples = 100

    # Fine-tuning epochs (increase for Activity 2).
    num_train_epochs = 5

    print("Loading model and dataset...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

    # GLUE SST-2: binary sentiment classification.
    hf_dataset = load_dataset("glue", "sst2")
    train_raw = hf_dataset["train"]
    val_raw = hf_dataset["validation"]

    # TextAttack expects a list of (text, label) pairs.
    from textattack.datasets import Dataset

    sample_for_attack = list(zip(
        train_raw["sentence"][:attack_examples],
        train_raw["label"][:attack_examples],
    ))
    textattack_dataset = Dataset(sample_for_attack)

    print("Generating adversarial examples...")
    adv_examples = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )

    print(f"{len(adv_examples)} adversarial examples generated.")

    # Combine clean and adversarial samples for adversarial training.
    clean_data = [
        {"text": x, "label": y}
        for x, y in zip(
            train_raw["sentence"][:clean_train_examples],
            train_raw["label"][:clean_train_examples],
        )
    ]
    combined_data = clean_data + adv_examples

    train_dataset = CustomDataset(combined_data, tokenizer)
    val_dataset = CustomDataset(
        [
            {"text": x, "label": y}
            for x, y in zip(
                val_raw["sentence"][:val_examples],
                val_raw["label"][:val_examples],
            )
        ],
        tokenizer,
    )

    print("Starting fine-tuning...")

    args = TrainingArguments(
        output_dir="./defended_model",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        logging_dir="./logs",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    trainer.train()

    # Persist the defended model locally.
    model.save_pretrained("./defended_model")
    tokenizer.save_pretrained("./defended_model")
    print("Fine-tuned model saved to ./defended_model.")

    # Re-wrap the fine-tuned model and rerun the attack to see whether robustness improved.
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
    _ = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )


if __name__ == "__main__":
    main()


Loading model and dataset...


[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Generating adversarial examples...
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 15 / 4 / 1 / 20: 100%|██████████| 20/20 [00:09<00:00,  2.01it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 15     |
| Number of failed attacks:     | 4      |
| Number of skipped attacks:    | 1      |
| Original accuracy:            | 95.0%  |
| Accuracy under attack:        | 20.0%  |
| Attack success rate:          | 78.95% |
| Average perturbed word %:     | 26.23% |
| Average num. words per input: | 8.7    |
| Avg num queries:              | 68.42  |
+-------------------------------+--------+



Example 1
Original text: with his usual intelligence and subtlety 
Perturbed text: with his usual tidings and subtlety 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 2
Original text: on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 
Perturbed text: on the forged revenge-of-the-nerds clichés the filmmakers could dredge up 
Ground-truth label: 0
Predicted label after attack: 0
Attack successful: False

Example 3
Original text: remains utterly satisfied to remain the same throughout 
Perturbed text: remains perfectly satisfied to remain the same throughout 
Ground-truth label: 0
Predicted label after attack: 1
Attack successful: True

Example 4
Original text: that 's far too tragic to merit such superficial treatment 
Perturbed text: that 's ALIR too tragical to deservingness such trivial intervention 
Ground-truth label: 0
Predicted label after attack: 0
Attack successful: False

Example 5
Original text: for those moviego

[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Fine-tuned model saved to ./defended_model.
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 5 / 15 / 0 / 20: 100%|██████████| 20/20 [00:13<00:00,  1.44it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 5      |
| Number of failed attacks:     | 15     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 75.0%  |
| Attack success rate:          | 25.0%  |
| Average perturbed word %:     | 41.18% |
| Average num. words per input: | 8.7    |
| Avg num queries:              | 86.45  |
+-------------------------------+--------+

Example 1
Original text: with his usual intelligence and subtlety 
Perturbed text: with his usual news and refinement 
Ground-truth label: 1
Predicted label after attack: 1
Attack successful: False

Example 2
Original text: on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 
Perturbed text: on the speculative revenge-of-the-nerds clichés the filmmakers could dredge up 
Ground-truth label: 0
Predicted label after a

# Activities

1. Increase the number of examples. What are your conclusions about the model's performance? Does the model improve or get worse? *Hint: Example Quantity*

2. Increase the number of training epochs. Is there any improvement in the model?

3.  Change the attacker to DeepWordBugGao2018. What are the main differences compared to PWWSRen2019? Which attacker is more difficult to correct?

4. Change the model to cardiffnlp/twitter-roberta-base-sentiment-latest. What is the performance? Write your conclusions about the whole process.